# Exercise 9 - Efficient pipeline on Fashion MNIST dataset

In [29]:
import os
import math
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.utils import shuffle
import pathlib
import tarfile
import urllib.request
import tensorflow_datasets as tfds

In [2]:
keras.utils.set_random_seed(42)

(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

X_train_full = X_train_full[..., np.newaxis]
X_test = X_test[..., np.newaxis]

X_train, y_train = X_train_full[5000:], y_train_full[5000:]
X_val, y_val = X_train_full[:5000], y_train_full[:5000]

print(X_train.shape, X_val.shape, X_test.shape)

(55000, 28, 28, 1) (5000, 28, 28, 1) (10000, 28, 28, 1)


In [4]:
def serialize_example(image, label):
    image_tensor = tf.io.serialize_tensor(image)
    feature = {"image": tf.train.Feature(bytes_list=tf.train.BytesList(value=[image_tensor.numpy()])), 
               "label": tf.train.Feature(int64_list=tf.train.Int64List(value=[int(label)]))}
    example = tf.train.Example(features=tf.train.Features(feature=feature))
    return example.SerializeToString()

def write_tfrecords(images, labels, output_dir, prefix, n_shards=10):
    os.makedirs(output_dir, exist_ok=True)
    images, labels = shuffle(images, labels, random_state=42)
    n_per_shard = math.ceil(len(images) / n_shards)

    for shard_id in range(n_shards):
        start = shard_id * n_per_shard
        end = min(start + n_per_shard, len(images))
        if start >= end:
            break

        filename = os.path.join(output_dir, f"{prefix}-{shard_id:05d}.tfrecord")
        with tf.io.TFRecordWriter(filename) as writer:
            for image, label in zip(images[start:end], labels[start:end]):
                writer.write(serialize_example(image, label))

write_tfrecords(X_train, y_train, "fashion_tfrecords", "train")
write_tfrecords(X_val, y_val, "fashion_tfrecords", "val", n_shards=2)
write_tfrecords(X_test, y_test, "fashion_tfrecords", "test", n_shards=2)

In [5]:
feature_description = {"image": tf.io.FixedLenFeature([], tf.string), "label": tf.io.FixedLenFeature([], tf.int64)}

def parse_example(serialized):
    example = tf.io.parse_single_example(serialized, feature_description)
    image = tf.io.parse_tensor(example["image"], out_type=tf.uint8)
    image = tf.ensure_shape(image, [28, 28, 1])
    label = example["label"]
    return image, label

In [6]:
AUTOTUNE = tf.data.AUTOTUNE
BATCH_SIZE = 32

def make_dataset(pattern, training=False):
    files = tf.data.Dataset.list_files(pattern, shuffle=training, seed=42)
    dataset = files.interleave(lambda filename: tf.data.TFRecordDataset(filename), num_parallel_calls=AUTOTUNE, cycle_length=AUTOTUNE,
        deterministic=not training)
    dataset = dataset.map(parse_example, num_parallel_calls=AUTOTUNE)

    if training:
        dataset = dataset.shuffle(10_000)

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTOTUNE)
    return dataset

train_dataset = make_dataset("fashion_tfrecords/train-*.tfrecord", training=True)
val_dataset = make_dataset("fashion_tfrecords/val-*.tfrecord")
test_dataset = make_dataset("fashion_tfrecords/test-*.tfrecord")

In [7]:
normalization = keras.layers.Normalization()
normalization.adapt(train_dataset.map(lambda image, label: tf.cast(image, tf.float32) / 255.))

model = keras.Sequential([
    keras.layers.Input(shape=(28, 28, 1), dtype=tf.uint8),
    keras.layers.Rescaling(1. / 255),
    normalization,
    keras.layers.Flatten(),
    keras.layers.Dense(300, activation="relu"),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(10, activation="softmax")
])

model.compile(optimizer=keras.optimizers.Nadam(), loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [8]:
log_dir = "logs/fashion_mnist"
tensorboard_callback = keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1, profile_batch=(10, 20))
early_stopping = keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True)
history = model.fit(train_dataset, validation_data=val_dataset, epochs=20, callbacks=[tensorboard_callback, early_stopping])

C:\Users\sumuk\anaconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/20
   1716/Unknown 31s 14ms/step - accuracy: 0.8376 - loss: 0.4448 

C:\Users\sumuk\anaconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


1719/1719 ━━━━━━━━━━━━━━━━━━━━ 33s 15ms/step - accuracy: 0.8377 - loss: 0.4445 - val_accuracy: 0.8658 - val_loss: 0.3654
Epoch 2/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 27s 15ms/step - accuracy: 0.8719 - loss: 0.3459 - val_accuracy: 0.8756 - val_loss: 0.3386
Epoch 3/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 27s 15ms/step - accuracy: 0.8841 - loss: 0.3088 - val_accuracy: 0.8836 - val_loss: 0.3263
Epoch 4/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 29s 17ms/step - accuracy: 0.8936 - loss: 0.2871 - val_accuracy: 0.8902 - val_loss: 0.3054
Epoch 5/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 28s 16ms/step - accuracy: 0.8990 - loss: 0.2694 - val_accuracy: 0.8856 - val_loss: 0.3204
Epoch 6/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 26s 15ms/step - accuracy: 0.9042 - loss: 0.2524 - val_accuracy: 0.8876 - val_loss: 0.3131
Epoch 7/20
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 25s 14ms/step - accuracy: 0.9091 - loss: 0.2398 - val_accuracy: 0.8864 - val_loss: 0.3202


In [9]:
%load_ext tensorboard
%tensorboard --logdir logs/fashion_mnist

In [10]:
test_loss, test_accuracy = model.evaluate(test_dataset, verbose=0)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

Test loss: 0.3406
Test accuracy: 0.8788


C:\Users\sumuk\anaconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


# Exercise 10 - Efficient pipeline on IMDb dataset

In [19]:
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.imdb.load_data(path="imdb.npz", num_words=10_000)

print("Successfully loaded local .npz dataset!")
print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

word_index = tf.keras.datasets.imdb.get_word_index()
index_word = {index + 3: word for word, index in word_index.items()}

# Add special tokens
index_word[0] = "<PAD>"
index_word[1] = "<START>"
index_word[2] = "<UNK>"
index_word[3] = "<UNUSED>"

def decode_reviews(sequences):
    return [" ".join(index_word.get(token, "<UNK>") for token in review if token != 0) for review in sequences]

X_train = decode_reviews(X_train)
X_test = decode_reviews(X_test)

rng = np.random.default_rng(seed=42)
indices = rng.permutation(len(X_test))

X_test = [X_test[i] for i in indices]
y_test = y_test[indices]

X_valid = X_test[:15_000]
y_valid = y_test[:15_000]

X_test = X_test[15_000:]
y_test = y_test[15_000:]

print("\nFinal dataset sizes:")
print("Training:", len(X_train))
print("Validation:", len(X_valid))
print("Test:", len(X_test))

Successfully loaded local .npz dataset!
Training data shape: (25000,)
Testing data shape: (25000,)
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step

Final dataset sizes:
Training: 25000
Validation: 15000
Test: 10000


In [20]:
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(reviews, labels, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((reviews, labels))

    if training:
        dataset = dataset.shuffle(buffer_size=len(reviews), seed=42)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTOTUNE)

    return dataset

train_ds = make_dataset(X_train, y_train, training=True)
valid_ds = make_dataset(X_valid, y_valid)
test_ds = make_dataset(X_test, y_test)

In [21]:
MAX_TOKENS = 10_000
SEQUENCE_LENGTH = 300
vectorizer = tf.keras.layers.TextVectorization(max_tokens=MAX_TOKENS, output_mode="int", output_sequence_length=SEQUENCE_LENGTH)

text_only_ds = train_ds.map(lambda review, label: review)
vectorizer.adapt(text_only_ds)

In [22]:
vocabulary = vectorizer.get_vocabulary()

print("Vocabulary size:", len(vocabulary))
print(vocabulary[:20])

Vocabulary size: 9909
['', '[UNK]', np.str_('unk'), np.str_('the'), np.str_('and'), np.str_('a'), np.str_('of'), np.str_('to'), np.str_('is'), np.str_('br'), np.str_('in'), np.str_('it'), np.str_('i'), np.str_('this'), np.str_('that'), np.str_('was'), np.str_('as'), np.str_('for'), np.str_('with'), np.str_('movie')]


In [23]:
class MeanEmbedding(tf.keras.layers.Layer):
    def __init__(self, input_dim, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.embedding = tf.keras.layers.Embedding(input_dim=input_dim, output_dim=output_dim, mask_zero=True)

    def call(self, inputs):
        embeddings = self.embedding(inputs)
        mask = tf.cast(inputs != 0, tf.float32)
        mask = tf.expand_dims(mask, axis=-1)
        sum_embeddings = tf.reduce_sum(embeddings * mask, axis=1)
        word_counts = tf.reduce_sum(mask, axis=1)
        mean_embeddings = tf.math.divide_no_nan(sum_embeddings, word_counts)

        return mean_embeddings * tf.sqrt(word_counts)

In [24]:
EMBEDDING_DIM = 64

model = tf.keras.Sequential([tf.keras.Input(shape=[], dtype=tf.string), vectorizer, MeanEmbedding(input_dim=len(vocabulary), output_dim=EMBEDDING_DIM),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ text_vectorization                   │ (None, 300)                 │               0 │
│ (TextVectorization)                  │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ mean_embedding (MeanEmbedding)       │ (None, 64)                  │         634,176 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 64)                  │           4,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 638,401 (2.44 MB)

 Trainable params: 638,401 (2.44 MB)

 Non-trainable params: 0 (0.00 B)

In [25]:
model.compile(loss="binary_crossentropy", optimizer="nadam", metrics=["accuracy"])
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2, restore_best_weights=True)
history = model.fit(train_ds, validation_data=valid_ds, epochs=10, callbacks=[early_stopping])

Epoch 1/10


C:\Users\sumuk\anaconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


782/782 ━━━━━━━━━━━━━━━━━━━━ 28s 30ms/step - accuracy: 0.8462 - loss: 0.3519 - val_accuracy: 0.8748 - val_loss: 0.2969
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 22s 29ms/step - accuracy: 0.9190 - loss: 0.2121 - val_accuracy: 0.8579 - val_loss: 0.3414
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 23s 30ms/step - accuracy: 0.9380 - loss: 0.1618 - val_accuracy: 0.8600 - val_loss: 0.3706


In [26]:
test_loss, test_accuracy = model.evaluate(test_ds)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.8758 - loss: 0.2937
Test loss: 0.29370132088661194
Test accuracy: 0.8758000135421753
